# Variable objetivo
### Mg. Ing. Diego Martín Méndez

In [1]:
import requests
import pandas as pd
import numpy as np
import time
from dotenv import load_dotenv
import os
from datetime import datetime
from pathlib import Path

In [2]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas pandas-market-calendars

Note: you may need to restart the kernel to use updated packages.


In [4]:
# ----------------------------
# 1. Configuración
# ----------------------------
class FMPConfig: 
    def __init__(self, env_file: str = ".env"):
        load_dotenv(dotenv_path=env_file)
        self.api_key = os.getenv("API_KEY")

    def get_api_key(self):
        if not self.api_key:
            raise ValueError("No se encontró 'API_KEY' en el archivo .env ni en las variables de entorno.")
        return self.api_key

In [5]:
# ----------------------------
# 2. Cliente general
# ----------------------------
class FMPEndpointStable:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable/"

    def get(self, endpoint: str, params: dict = None):
        if params is None:
            params = {}
        params["apikey"] = self.api_key
        url = self.base_url + endpoint
        response = requests.get(url, params=params)

        if response.status_code != 200:
            raise Exception(f"Error {response.status_code}: {response.text}")

        return response.json()

class FMPEndpointV3:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/api/v3/"

    def get(self, endpoint: str, params: dict = None):
        if params is None:
            params = {}
        params["apikey"] = self.api_key
        url = self.base_url + endpoint
        response = requests.get(url, params=params)

        if response.status_code != 200:
            raise Exception(f"Error {response.status_code}: {response.text}")

        return response.json()

In [6]:
income_statement_quarter_nyse_nasdaq = pd.read_parquet("archivos/income_statements_quarter_nyse_nasdaq.parquet")
income_statement_quarter_nyse_nasdaq

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,...,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-30,BTGO,USD,0000000000,2025-09-30,2025-09-30 00:00:00,2025,Q3,5.810456e+09,5.760237e+09,...,2.267200e+07,0.0,-15835000.0,6.837000e+06,0.0,6.837000e+06,0.0000,0.0000,0.000000e+00,0.000000e+00
1,2024-12-31,BTGO,USD,0000000000,2024-12-31,2024-12-31 00:00:00,2024,Q4,1.140324e+09,1.120136e+09,...,1.294010e+08,0.0,-80411000.0,4.899000e+07,0.0,4.899000e+07,0.0000,0.0000,0.000000e+00,0.000000e+00
2,2024-09-30,BTGO,USD,0000000000,2024-09-30,2024-09-30 00:00:00,2024,Q3,8.178260e+08,8.060250e+08,...,-3.752000e+06,0.0,471000.0,-3.281000e+06,0.0,-3.281000e+06,0.0000,0.0000,0.000000e+00,0.000000e+00
3,2025-12-31,XOM,USD,0000034088,2026-01-30,2026-01-30 06:31:24,2025,Q4,8.003900e+10,6.492300e+10,...,6.609000e+09,0.0,0.0,6.501000e+09,0.0,6.501000e+09,1.5000,1.5300,4.331000e+09,4.238000e+09
4,2025-09-30,XOM,USD,0000034088,2025-11-03,2025-11-03 12:45:47,2025,Q3,8.333100e+10,6.464600e+10,...,7.768000e+09,0.0,0.0,7.548000e+09,0.0,7.548000e+09,1.7600,1.7600,4.331000e+09,4.331000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639941,2021-06-30,VHNAW,USD,0001868640,2021-06-30,2021-06-28 20:00:00,2022,Q1,0.000000e+00,0.000000e+00,...,-1.065600e+04,0.0,0.0,-1.065600e+04,0.0,-1.065600e+04,-0.0004,-0.0004,2.501250e+07,2.501250e+07
639942,2023-03-31,AFTR-WT,USD,0001865975,2023-05-04,2023-05-04 16:31:39,2023,Q1,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.303400e+05,0.0,2.303400e+05,-0.4000,-0.4000,6.250000e+06,6.250000e+06
639943,2022-09-30,AFTR-WT,USD,0001865975,2022-11-01,2022-11-01 16:30:57,2022,Q3,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.144000e+06,0.0,2.144000e+06,0.3200,0.3200,1.250000e+07,1.250000e+07
639944,2022-03-31,AFTR-WT,USD,0001865975,2022-05-05,2022-05-05 16:51:30,2022,Q1,0.000000e+00,0.000000e+00,...,0.000000e+00,0.0,0.0,2.577000e+06,0.0,2.577000e+06,0.1000,0.1000,2.500000e+07,2.500000e+07


In [7]:
income_statement_quarter_nyse_nasdaq["symbol"].nunique()

12289

In [8]:
income_statement_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 639946 entries, 0 to 639945
Data columns (total 39 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   date                                     639946 non-null  object 
 1   symbol                                   639946 non-null  object 
 2   reportedCurrency                         639946 non-null  object 
 3   cik                                      639946 non-null  object 
 4   filingDate                               639932 non-null  object 
 5   acceptedDate                             639932 non-null  object 
 6   fiscalYear                               639946 non-null  object 
 7   period                                   639946 non-null  object 
 8   revenue                                  639943 non-null  float64
 9   costOfRevenue                            639946 non-null  float64
 10  grossProfit                     

In [9]:
income_statement_quarter_nyse_nasdaq.isna().sum().sort_values(ascending=False)

weightedAverageShsOutDil                   8117
weightedAverageShsOut                      8117
netIncomeDeductions                        5007
otherAdjustmentsToNetIncome                4975
netIncomeFromContinuingOperations          1593
interestExpense                             590
interestIncome                              496
bottomLineNetIncome                         126
epsDiluted                                   41
eps                                          37
filingDate                                   14
acceptedDate                                 14
sellingAndMarketingExpenses                  13
generalAndAdministrativeExpenses             12
netIncomeFromDiscontinuedOperations          12
researchAndDevelopmentExpenses               10
nonOperatingIncomeExcludingInterest           5
incomeTaxExpense                              4
revenue                                       3
ebit                                          3
incomeBeforeTax                         

In [10]:
import re
import pandas as pd
import pandas_market_calendars as mcal

In [11]:
import re
import pandas as pd
import pandas_market_calendars as mcal


def crear_start_end_date_df(
    input_parquet: str = "archivos/income_statements_quarter_nyse_nasdaq.parquet"
) -> pd.DataFrame:
    
    def tiene_hora_explicita(x):
        if pd.isna(x):
            return False
        return bool(re.search(r"\b\d{1,2}:\d{2}(:\d{2})?\b", str(x)))

    def next_trading_day(d, trading_days):
        pos = trading_days.searchsorted(d, side="right")
        return trading_days[pos] if pos < len(trading_days) else pd.NaT

    def previous_trading_day(d, trading_days):
        pos = trading_days.searchsorted(d, side="left") - 1
        return trading_days[pos] if pos >= 0 else pd.NaT

    df = pd.read_parquet(input_parquet, columns=["symbol", "acceptedDate"]).copy()
    df["symbol"] = df["symbol"].astype(str)
    df["acceptedDate_ts"] = pd.to_datetime(df["acceptedDate"], errors="coerce")

    fechas_validas = df["acceptedDate_ts"].dropna()
    start = fechas_validas.min().date() - pd.Timedelta(days=10)
    end = fechas_validas.max().date() + pd.Timedelta(days=10)

    us_market = mcal.get_calendar("NYSE")
    schedule = us_market.schedule(start_date=start, end_date=end)
    trading_days = pd.Index(pd.to_datetime(schedule.index).date)
    trading_days_set = set(trading_days)

    start_dates = []
    close_time = pd.to_datetime("16:00:00").time()

    for row in df.itertuples(index=False):
        ts = row.acceptedDate_ts
        raw = row.acceptedDate

        if pd.isna(ts):
            start_dates.append(pd.NaT)
            continue

        d = ts.date()
        h = ts.time()

        if not tiene_hora_explicita(raw):
            start_dates.append(next_trading_day(d, trading_days))
            continue

        if h.hour == 0 and h.minute == 0 and h.second == 0:
            start_dates.append(next_trading_day(d, trading_days))
            continue

        if h < close_time:
            if d in trading_days_set:
                start_dates.append(d)
            else:
                start_dates.append(next_trading_day(d, trading_days))
        else:
            start_dates.append(next_trading_day(d, trading_days))

    df["start_date"] = start_dates

    df = df.sort_values(["symbol", "start_date", "acceptedDate"], na_position="last").reset_index(drop=True)

    df["next_start_date"] = df.groupby("symbol")["start_date"].shift(-1)

    df["end_date"] = df["next_start_date"].apply(
        lambda x: previous_trading_day(x, trading_days) if pd.notna(x) else pd.NaT
    )
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
    df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")
    df["acceptedDate"] = pd.to_datetime(df["acceptedDate"], errors="coerce")
    
    return df[["symbol", "acceptedDate", "start_date", "end_date"]]

In [12]:
symbol_dates = crear_start_end_date_df()

In [13]:
symbol_dates.head(10)

,symbol,acceptedDate,start_date,end_date
0,A,1999-01-31,1999-02-01,1999-04-30
1,A,1999-04-30,1999-05-03,1999-07-30
2,A,1999-07-31,1999-08-02,2000-01-25
3,A,2000-01-25,2000-01-26,2000-03-15
4,A,2000-03-15,2000-03-16,2000-06-12
5,A,2000-06-12,2000-06-13,2000-09-01
6,A,2000-09-01,2000-09-05,2001-01-17
7,A,2001-01-17,2001-01-18,2001-03-19
8,A,2001-03-19,2001-03-20,2001-06-14
9,A,2001-06-14,2001-06-15,2001-09-10


In [14]:
symbol_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 639946 entries, 0 to 639945
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   symbol        639946 non-null  object        
 1   acceptedDate  639932 non-null  datetime64[ns]
 2   start_date    639932 non-null  datetime64[ns]
 3   end_date      627643 non-null  datetime64[ns]
dtypes: datetime64[ns](3), object(1)
memory usage: 19.5+ MB


In [15]:
def agregar_adjclose_start_end(symbol_dates: pd.DataFrame) -> pd.DataFrame:

    config = FMPConfig()
    api_key = config.get_api_key()
    client = FMPEndpointV3(api_key)

    df = symbol_dates.copy()

    df["symbol"] = df["symbol"].astype(str)
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
    df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")

    df["adjClose_start_date"] = pd.NA
    df["adjClose_end_date"] = pd.NA

    symbols = sorted(df["symbol"].dropna().unique())

    for symbol in symbols:

        df_symbol = df[df["symbol"] == symbol]

        fechas = pd.concat([
            df_symbol["start_date"],
            df_symbol["end_date"]
        ]).dropna()

        if fechas.empty:
            continue

        fecha_min = fechas.min().strftime("%Y-%m-%d")
        fecha_max = fechas.max().strftime("%Y-%m-%d")

        data = client.get(
            f"historical-price-full/{symbol}",
            params={
                "from": fecha_min,
                "to": fecha_max
            }
        )

        if "historical" not in data:
            continue

        px = pd.DataFrame(data["historical"])

        if px.empty:
            continue

        px["date"] = pd.to_datetime(px["date"], errors="coerce").dt.strftime("%Y-%m-%d")
        px["adjClose"] = pd.to_numeric(px["adjClose"], errors="coerce")

        price_map = px.set_index("date")["adjClose"].to_dict()

        mask = df["symbol"] == symbol

        start_keys = df.loc[mask, "start_date"].dt.strftime("%Y-%m-%d")
        end_keys = df.loc[mask, "end_date"].dt.strftime("%Y-%m-%d")

        df.loc[mask, "adjClose_start_date"] = start_keys.map(price_map)
        df.loc[mask, "adjClose_end_date"] = end_keys.map(price_map)

    return df

In [ ]:
adjclose_start_end = agregar_adjclose_start_end(symbol_dates)

In [ ]:
adjclose_start_end.info()

In [ ]:
adjclose_start_end.to_parquet("archivos/adjclose_start_end.parquet", index=False)

In [17]:
adjclose_start_end = pd.read_parquet("archivos/adjclose_start_end.parquet")
adjclose_start_end

,symbol,acceptedDate,start_date,end_date,adjClose_start_date,adjClose_end_date
0,A,1999-01-31 00:00:00,1999-02-01,1999-04-30,NaN,NaN
1,A,1999-04-30 00:00:00,1999-05-03,1999-07-30,NaN,NaN
2,A,1999-07-31 00:00:00,1999-08-02,2000-01-25,NaN,37.8800
3,A,2000-01-25 00:00:00,2000-01-26,2000-03-15,38.24,63.9500
4,A,2000-03-15 00:00:00,2000-03-16,2000-06-12,66.09,36.9400
...,...,...,...,...,...,...
639941,ZYXI,2024-10-24 17:16:33,2024-10-25,2025-03-11,9.12,7.0000
639942,ZYXI,2025-03-11 16:45:24,2025-03-12,2025-04-29,3.41,2.2300
639943,ZYXI,2025-04-29 17:00:23,2025-04-30,2025-07-31,1.66,2.2300
639944,ZYXI,2025-07-31 17:15:53,2025-08-01,2025-11-17,1.26,0.5651


In [18]:
adjclose_start_end["target_return"] = (
    adjclose_start_end["adjClose_end_date"] / adjclose_start_end["adjClose_start_date"] - 1
)

In [19]:
adjclose_start_end

,symbol,acceptedDate,start_date,end_date,adjClose_start_date,adjClose_end_date,target_return
0,A,1999-01-31 00:00:00,1999-02-01,1999-04-30,NaN,NaN,NaN
1,A,1999-04-30 00:00:00,1999-05-03,1999-07-30,NaN,NaN,NaN
2,A,1999-07-31 00:00:00,1999-08-02,2000-01-25,NaN,37.8800,NaN
3,A,2000-01-25 00:00:00,2000-01-26,2000-03-15,38.24,63.9500,0.672333
4,A,2000-03-15 00:00:00,2000-03-16,2000-06-12,66.09,36.9400,-0.441065
...,...,...,...,...,...,...,...
639941,ZYXI,2024-10-24 17:16:33,2024-10-25,2025-03-11,9.12,7.0000,-0.232456
639942,ZYXI,2025-03-11 16:45:24,2025-03-12,2025-04-29,3.41,2.2300,-0.346041
639943,ZYXI,2025-04-29 17:00:23,2025-04-30,2025-07-31,1.66,2.2300,0.343373
639944,ZYXI,2025-07-31 17:15:53,2025-08-01,2025-11-17,1.26,0.5651,-0.551508


In [20]:
target_return = adjclose_start_end.drop(columns=["start_date", "end_date", "adjClose_start_date", "adjClose_end_date"])

In [21]:
target_return

,symbol,acceptedDate,target_return
0,A,1999-01-31 00:00:00,NaN
1,A,1999-04-30 00:00:00,NaN
2,A,1999-07-31 00:00:00,NaN
3,A,2000-01-25 00:00:00,0.672333
4,A,2000-03-15 00:00:00,-0.441065
...,...,...,...
639941,ZYXI,2024-10-24 17:16:33,-0.232456
639942,ZYXI,2025-03-11 16:45:24,-0.346041
639943,ZYXI,2025-04-29 17:00:23,0.343373
639944,ZYXI,2025-07-31 17:15:53,-0.551508


In [22]:
target_return = target_return.dropna(subset=["target_return"])

In [23]:
target_return

,symbol,acceptedDate,target_return
3,A,2000-01-25 00:00:00,0.672333
4,A,2000-03-15 00:00:00,-0.441065
5,A,2000-06-12 00:00:00,-0.105904
6,A,2000-09-01 00:00:00,0.002023
7,A,2001-01-17 00:00:00,-0.452671
...,...,...,...
639940,ZYXI,2024-07-25 17:00:55,-0.038074
639941,ZYXI,2024-10-24 17:16:33,-0.232456
639942,ZYXI,2025-03-11 16:45:24,-0.346041
639943,ZYXI,2025-04-29 17:00:23,0.343373


In [24]:
target_return.info()

<class 'pandas.core.frame.DataFrame'>
Index: 469029 entries, 3 to 639944
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   symbol         469029 non-null  object        
 1   acceptedDate   469029 non-null  datetime64[ns]
 2   target_return  469029 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 14.3+ MB


In [25]:
target_return.to_parquet("archivos/target_return.parquet", index=False)